<center>
<img src="https://drive.google.com/uc?id=1B7bif7ps19gO0Ieke_E-sOwSPMkfhWtf" width="80%">
</center>

# Rethinking the Training Loop (Part I)

**DCA0305 · Machine Learning Based Systems Design**

This notebook rebuilds the PyTorch training loop step by step. Along the way we will refactor a naive script into a modular pipeline built from four blocks. Each block has evolved through several versions during the course, and here we work with the most refined ones.

| Block | Version used here | What changed along the way |
|---|---|---|
| Data Generation | v0 | Plain NumPy, nothing fancy |
| Data Preparation | v2 | `TensorDataset` + `DataLoader` + `random_split` |
| Model Configuration | v2 | Higher-order functions build the train and validation steps |
| Model Training | v4 | Mini-batch loop + dataloader + validation function |

Two extra stages close the loop for real projects. Saving/loading checkpoints and making predictions with a trained model.


## 🎯 Learning objectives

By the end of this notebook you should be able to

1. Explain what a higher-order function is and why the training pipeline uses closures.
2. Convert NumPy arrays into PyTorch tensors and justify why they stay on the CPU until batch time.
3. Build a `Dataset`, split it reproducibly, and wrap it with `DataLoader`s.
4. Write `make_train_step_fn` and `make_val_step_fn` from memory and explain each of the four training steps.
5. Explain the difference between `model.train()` / `model.eval()` and `torch.no_grad()`.
6. Save a full training checkpoint, restore it, and resume training or run inference safely.

## How to get the most retention out of this notebook

Passive reading produces almost no long-term memory. This notebook is designed around three evidence-based habits.

- **Predict before you run.** Whenever you see a 🔮 *Predict* prompt, write your guess (mentally or in a scratch cell) before executing the code. Being wrong first is what makes the correct answer stick.
- **Retrieve, don't reread.** The ✅ *Check yourself* boxes ask questions whose answers are hidden. Try to answer out loud before clicking to reveal.
- **Space it out.** Come back to the 📝 *Final self-test* at the end of the notebook one or two days after class and try it with the notebook closed.


# 0. Warm-up · Higher-Order Functions

Before touching PyTorch, we need one Python concept that powers the whole refactoring in this lesson. A **higher-order function** is a function that *takes another function as an argument* or *returns a function as its result* (or both). This idea comes from functional programming and it is exactly what will let us package the training logic into a reusable "step function".

## Functions accepting functions

`map` is the classic example. It receives a function (`square`) and applies it to every element of an iterable.


In [ ]:
def square(x):
    return x * x

numbers = [1, 2, 3, 4, 5]
squares = map(square, numbers)   # square is passed WITHOUT parentheses (the function itself, not its result)
print(list(squares))             # [1, 4, 9, 16, 25]

## Functions returning functions (closures)

The pattern below is the one we will reuse. `multiplier` is a *factory*. It builds and returns a customized `inner` function. Crucially, `inner` remembers the value of `n` even after `multiplier` has finished running. A function that captures variables from its enclosing scope like this is called a **closure**.


In [ ]:
def multiplier(n):
    def inner(x):
        return x * n     # `n` is captured from the enclosing scope
    return inner         # we return the FUNCTION, we do not call it

double = multiplier(2)   # double is a function with n=2 "baked in"
triple = multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15
print(type(double))

### 🔮 Predict

Before running the next cell, answer mentally. If we now call `multiplier(10)(4)`, what is printed? Why does chaining two pairs of parentheses work at all?


In [ ]:
# The first call returns a function; the second pair of parentheses calls that returned function
print(multiplier(10)(4))

### Why do we care?

Later in this notebook, `make_train_step_fn(model, loss_fn, optimizer)` will return a function `perform_train_step_fn(x, y)` with the model, the loss, and the optimizer *baked in*. The training loop then becomes beautifully simple. It only needs to feed data into that returned function, without dragging the model, loss, and optimizer around as arguments everywhere.

✅ **Check yourself**

<details><summary>1. What are the two defining behaviors of a higher-order function?</summary>

It takes one or more functions as arguments, or it returns a function as its result (or both). `map(square, numbers)` illustrates the first behavior and `multiplier(n)` illustrates the second.
</details>

<details><summary>2. In <code>double = multiplier(2)</code>, why do we write <code>multiplier(2)</code> with parentheses but pass <code>square</code> to <code>map</code> without parentheses?</summary>

`multiplier(2)` must be *called* so that it manufactures and returns the inner function. `square` is passed without parentheses because `map` needs the function object itself, so it can call it later on each element. Parentheses mean "call now", no parentheses mean "hand over the function".
</details>

<details><summary>3. After <code>multiplier</code> returns, its local variable <code>n</code> should be gone. Why does <code>double(5)</code> still work?</summary>

Because `inner` is a closure. Python keeps the captured variables alive in the function's `__closure__` as long as the returned function exists. You can inspect it with `double.__closure__[0].cell_contents`, which returns `2`.
</details>


# 1. Import libraries

A quick tour of what each import is for.

- `numpy` generates the synthetic data and averages mini-batch losses.
- `sklearn.linear_model.LinearRegression` will serve as a *ground-truth solver*. Since our model is a plain linear regression, sklearn's closed-form solution gives us the answer PyTorch should converge to. Comparing both is a great sanity check.
- `torch`, `torch.nn`, `torch.optim` are the core PyTorch pieces (tensors, layers/losses, optimizers).
- `Dataset`, `TensorDataset`, `DataLoader`, `random_split` are the data pipeline utilities from `torch.utils.data`.
- `matplotlib` plots the loss curves.


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torch.utils.data.dataset import random_split

import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('fivethirtyeight')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

# 2. Data Generation (v0)

We create synthetic data from a *known* linear model so we can later verify whether training recovered the true parameters.

$$y = b + w x + \epsilon, \qquad b = 1, \; w = 2, \; \epsilon \sim \mathrm{N}(0, 0.1^2)$$

Three details deserve attention.

1. **`np.random.seed(42)`** makes the pseudo-random generator deterministic, so every run (and every student) produces the same data. Reproducibility is a first-class engineering requirement, not a cosmetic detail.
2. **Shapes.** `x` and `y` are `(N, 1)` column vectors, not flat `(N,)` arrays. PyTorch layers expect the first dimension to be the *batch* dimension, so keeping data 2-D from the start avoids painful broadcasting bugs later.
3. **The noise term** `0.1 * np.random.randn(N, 1)` is what makes this a *statistics* problem instead of an algebra problem. Without noise, two points would suffice to determine the line exactly.


In [ ]:
true_b = 1
true_w = 2
N = 100

# Data Generation
np.random.seed(42)
x = np.random.rand(N, 1)                                  # uniform in [0, 1)
y = true_b + true_w * x + (.1 * np.random.randn(N, 1))    # linear model + gaussian noise

print(x.shape, y.shape)
print(x[:3].ravel(), y[:3].ravel())

In [ ]:
# Visualizing the raw data. A picture beats any print.
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x, y, alpha=.6)
ax.plot([0, 1], [true_b, true_b + true_w], 'k--', lw=2, label='true line  y = 1 + 2x')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend()
plt.tight_layout()
plt.show()

✅ **Check yourself**

<details><summary>1. Why do we keep <code>x</code> with shape <code>(100, 1)</code> instead of <code>(100,)</code>?</summary>

Because PyTorch's `nn.Linear(1, 1)` expects inputs shaped `(batch_size, in_features)`. A flat `(100,)` array would be interpreted ambiguously and typically triggers shape or broadcasting errors when computing the loss (e.g. a `(100,)` prediction against a `(100, 1)` target silently broadcasts to `(100, 100)` in some frameworks). Explicit 2-D shapes prevent an entire class of bugs.
</details>

<details><summary>2. If we removed the seed, would training still work? What exactly would change?</summary>

Training would still work. The data would simply be different on every run, so losses, learned parameters, and plots would not be exactly reproducible. Nothing conceptual changes, but debugging and grading become much harder.
</details>


# 3. Data Preparation (v2)

## 3.1 From NumPy to tensors, and the CPU question

`torch.from_numpy(x)` creates a tensor that **shares memory** with the NumPy array (zero copy). `.float()` then converts it to `float32`, which *does* copy, because NumPy defaults to `float64` while PyTorch models default to `float32`.

The slide asks the key question. *"Wait, is this a CPU tensor now? Why? Where is `.to(device)`?"*

The answer defines the whole v2 design. **The dataset stays on the CPU on purpose.** In real projects, datasets are far larger than GPU memory (think ImageNet, not 100 points). The strategy is therefore

1. keep the *full* dataset on CPU (or even on disk),
2. let the `DataLoader` assemble small mini-batches,
3. move **only the current mini-batch** to the GPU inside the training loop.

Only the *model* lives permanently on the GPU. Data visits it batch by batch.

## 3.2 `Dataset` · the abstraction

A PyTorch `Dataset` is any object that answers two questions. *"How many samples do you have?"* (`__len__`) and *"give me sample number i"* (`__getitem__`). Here is the custom implementation shown in the slides, for reference. It is worth reading once so `TensorDataset` does not look like magic.


In [ ]:
class CustomDataset(Dataset):
    def __init__(self, x_tensor, y_tensor):
        self.x = x_tensor
        self.y = y_tensor

    def __getitem__(self, index):
        return (self.x[index], self.y[index])

    def __len__(self):
        return len(self.x)

# Quick demo (this class works, but below we will use TensorDataset, which does exactly this)
_demo = CustomDataset(torch.from_numpy(x).float(), torch.from_numpy(y).float())
print(len(_demo))
print(_demo[0])   # a (feature, label) tuple

## 3.3 `TensorDataset`, `random_split`, and `DataLoader`

Since our custom class merely indexes two tensors in parallel, PyTorch already ships it as **`TensorDataset`**.

**Split BEFORE anything else touches the data.** We build a dataset with *all* points and only then split into train/validation. The validation set must simulate unseen data, so any statistic or decision derived from it would be a **data leak**.

**Improvement over the original notebook.** `random_split` uses PyTorch's global RNG, so its result depends on whatever consumed random numbers before it. Passing an explicit `generator` pins the split itself, independently of everything else. This is the recommended practice for reproducible experiments.

**`DataLoader`** turns a dataset into an iterable of mini-batches. Note the asymmetry.

- `shuffle=True` for training. Reshuffling every epoch decorrelates consecutive batches and improves SGD.
- No shuffle for validation. Order does not matter for evaluation, and a fixed order makes runs comparable.


In [ ]:
torch.manual_seed(13)

# Builds tensors from numpy arrays BEFORE split (CPU tensors, on purpose!)
x_tensor = torch.from_numpy(x).float()
y_tensor = torch.from_numpy(y).float()

# Builds dataset containing ALL data points
dataset = TensorDataset(x_tensor, y_tensor)

# Performs the split
ratio = .8
n_total = len(dataset)
n_train = int(n_total * ratio)
n_val = n_total - n_train

# IMPROVEMENT: an explicit generator makes the split reproducible on its own,
# no matter what other code consumed random numbers before this cell
split_generator = torch.Generator().manual_seed(42)
train_data, val_data = random_split(dataset, [n_train, n_val], generator=split_generator)

# Builds a loader for each set
train_loader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(dataset=val_data, batch_size=16)

print(f"train: {len(train_data)} samples | val: {len(val_data)} samples")

### 🔮 Predict

With 80 training samples and `batch_size=16`, how many mini-batches does one training epoch contain? And with 20 validation samples? Compute both before running the next cell.


In [ ]:
print(f"train batches per epoch: {len(train_loader)}")
print(f"val batches per epoch:   {len(val_loader)}")

# Peek at one batch to internalize shapes
xb, yb = next(iter(train_loader))
print(f"one batch -> x: {tuple(xb.shape)}, y: {tuple(yb.shape)}, device: {xb.device}")

✅ **Check yourself**

<details><summary>1. Why don't we call <code>.to(device)</code> on <code>x_tensor</code> and <code>y_tensor</code> right here?</summary>

Because real datasets do not fit in GPU memory. Keeping the dataset on CPU and shipping only the active mini-batch to the GPU scales to datasets of any size. The transfer happens inside the training loop, right before the step function is called.
</details>

<details><summary>2. What would go wrong if we split the data AFTER normalizing it with statistics computed on the full dataset?</summary>

The validation set would have influenced the preprocessing (its values contribute to the mean/std), which leaks information from "unseen" data into training. Validation metrics would then be optimistically biased. The rule is to fit any preprocessing on the training split only.
</details>

<details><summary>3. Why <code>shuffle=True</code> only on the train loader?</summary>

Shuffling changes which samples land in the same mini-batch each epoch, which reduces correlation between consecutive gradient estimates and helps SGD escape bad patterns (e.g. data sorted by label). Validation computes a fixed metric over all samples, so order is irrelevant and determinism is preferable.
</details>

<details><summary>4. <code>torch.from_numpy</code> vs <code>torch.as_tensor</code> vs <code>torch.tensor</code>. What is the difference?</summary>

`torch.from_numpy` always shares memory with the NumPy array and only accepts NumPy arrays. `torch.as_tensor` shares memory when possible (and accepts lists, scalars, tensors too). `torch.tensor` always copies. In this pipeline the subsequent `.float()` copies anyway, so the three are interchangeable here, but knowing the semantics matters when arrays are huge.
</details>


# 4. Model Configuration (v2)

## 4.1 The step-function factories

Here the higher-order-function warm-up pays off. Each factory below receives the *ingredients* (model, loss, optimizer) and returns a function that performs **one step over one mini-batch**. The returned functions are closures, so the training loop never needs to see the model or the optimizer again.

The four canonical steps of a training iteration, in order.

1. **Forward pass.** `yhat = model(x)` computes predictions.
2. **Loss.** `loss = loss_fn(yhat, y)` compares predictions to targets and builds the computation graph.
3. **Backward pass.** `loss.backward()` traverses that graph and *accumulates* `param.grad` for every parameter with `requires_grad=True`.
4. **Update.** `optimizer.step()` applies the update rule (plain SGD here, `param -= lr * param.grad`), then `optimizer.zero_grad()` clears the gradients.

Why clear the gradients at all? Because PyTorch **accumulates** gradients by design (useful for RNNs and gradient accumulation tricks). If you forget `zero_grad()`, every step adds new gradients on top of old ones and training silently diverges. This is one of the most common PyTorch bugs in the wild.

A subtle but important distinction. **`model.train()` is not what enables gradients.** It only switches layers such as `Dropout` and `BatchNorm` into training behavior. Our tiny linear model has neither, so the call changes nothing today, but writing it now builds the correct habit for every future model.


In [ ]:
def make_train_step_fn(model, loss_fn, optimizer):
    # Builds function that performs a step in the train loop
    def perform_train_step_fn(x, y):
        # Sets model to TRAIN mode (affects Dropout/BatchNorm; harmless here, essential later)
        model.train()

        # Step 1 - Computes our model's predicted output - forward pass
        yhat = model(x)
        # Step 2 - Computes the loss
        loss = loss_fn(yhat, y)
        # Step 3 - Computes gradients for both "b" and "w" parameters
        loss.backward()
        # Step 4 - Updates parameters using gradients and the learning rate,
        # then clears gradients (PyTorch ACCUMULATES them by default!)
        optimizer.step()
        optimizer.zero_grad()

        # Returns a plain Python float (detached from the graph)
        return loss.item()

    # Returns the function that will be called inside the train loop
    return perform_train_step_fn

## 4.2 The validation step factory

Validation answers one question only. *"How well does the current model generalize?"* So there is no backward pass and no update. Steps 3 and 4 disappear, and `model.eval()` switches Dropout/BatchNorm into inference behavior.

Note what is **not** here. `torch.no_grad()` is *not* inside this function. The caller (the training loop) wraps validation with it. Both placements work; the course keeps the context manager in the loop so the step function stays symmetric with the training one.


In [ ]:
def make_val_step_fn(model, loss_fn):
    # Builds function that performs a step in the validation loop
    def perform_val_step_fn(x, y):
        # Sets model to EVAL mode (affects Dropout/BatchNorm)
        model.eval()

        # Step 1 - Computes our model's predicted output - forward pass
        yhat = model(x)
        # Step 2 - Computes the loss
        loss = loss_fn(yhat, y)
        # There is no need for Steps 3 and 4, since we don't update parameters during evaluation
        return loss.item()

    return perform_val_step_fn

## 4.3 Assembling the configuration

Everything the *model* needs is defined here, in one place.

- **`device`** picks the GPU when available. The **model** is sent to it immediately (`Model lives on GPU`, as the slide says). The data will follow, one batch at a time.
- **`lr = 0.1`** is the learning rate ($\eta$). Too small and convergence crawls, too large and the loss oscillates or explodes. For this convex 2-parameter problem, 0.1 is comfortable.
- **`torch.manual_seed(42)`** right before model creation pins the random initialization of `nn.Linear`'s weight and bias.
- **`nn.Sequential(nn.Linear(1, 1))`** is our whole "network". One input feature, one output, so exactly two parameters (w and b), matching the data-generating equation.
- **`nn.MSELoss(reduction='mean')`** averages squared errors over the batch. Mean (rather than sum) keeps the gradient scale independent of batch size, so `lr` does not need retuning when the batch size changes.
- **`optim.SGD(model.parameters(), ...)`** receives the parameters *from the model itself*. This is what earlier versions of the pipeline did by hand with raw tensors.


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Sets learning rate - this is "eta" ~ the "n"-like Greek letter
lr = 0.1

torch.manual_seed(42)
# Creates the model and sends it at once to the device
model = nn.Sequential(nn.Linear(1, 1)).to(device)

# Defines an SGD optimizer to update the parameters (retrieved directly from the model)
optimizer = optim.SGD(model.parameters(), lr=lr)

# Defines an MSE loss function
loss_fn = nn.MSELoss(reduction='mean')

# Uses the factories to build both step functions (closures!)
train_step_fn = make_train_step_fn(model, loss_fn, optimizer)
val_step_fn = make_val_step_fn(model, loss_fn)

# Where does the model start from? Random init, nowhere near b=1, w=2
print(model.state_dict())

✅ **Check yourself**

<details><summary>1. Why does <code>perform_train_step_fn</code> return <code>loss.item()</code> instead of <code>loss</code>?</summary>

`loss` is a tensor attached to the computation graph. Returning and storing it every step would keep the whole graph alive in memory (a classic memory leak). `.item()` extracts a plain Python float, detached from the graph.
</details>

<details><summary>2. What happens if <code>optimizer.zero_grad()</code> is removed?</summary>

Gradients accumulate across steps, so each update uses the *sum* of all past gradients. The effective step grows without bound and the loss typically diverges. Try it later as an experiment. It is very instructive to watch.
</details>

<details><summary>3. <code>model.train()</code> vs <code>torch.no_grad()</code>. Which controls what?</summary>

They are orthogonal. `model.train()`/`model.eval()` toggles the *behavior of specific layers* (Dropout drops or not, BatchNorm uses batch or running statistics). `torch.no_grad()` disables *gradient tracking* to save memory and compute. Correct validation uses both `model.eval()` and `no_grad`.
</details>

<details><summary>4. Why does <code>make_val_step_fn</code> not receive the optimizer?</summary>

Because validation never updates parameters. Not passing the optimizer makes that guarantee structural. The function *could not* update the model even by accident.
</details>


# 5. Model Training (v4)

## 5.1 The mini-batch helper

The inner loop over batches is identical for training and validation. Only two things change, *which loader* and *which step function*. So we extract it into `mini_batch`, a higher-order function once again (it receives `step_fn` as an argument).

For each batch, it

1. moves the batch to the device (**this** is where data finally meets the GPU),
2. calls the step function,
3. collects the loss.

It returns the **average** loss across batches, one number per epoch.


In [ ]:
def mini_batch(device, data_loader, step_fn):
    mini_batch_losses = []
    for x_batch, y_batch in data_loader:
        # The data lives on the CPU; each mini-batch visits the device only when needed
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        mini_batch_loss = step_fn(x_batch, y_batch)
        mini_batch_losses.append(mini_batch_loss)

    loss = np.mean(mini_batch_losses)
    return loss

## 5.2 The training loop itself

Look how small the loop became after all the refactoring. One line for training, three for validation. That is the payoff of higher-order functions.

The validation block runs under **`torch.no_grad()`**. During evaluation we do not need gradients, so building the computation graph would waste memory and time. Disabling it also protects us from accidentally calling `.backward()` on validation data.

*Terminology check.* One **epoch** is one full pass over the training set (5 mini-batches of 16 here). One **step** is one mini-batch update. 200 epochs therefore mean 1000 gradient updates.


In [ ]:
%%time
# Defines number of epochs
n_epochs = 200

losses = []
val_losses = []

for epoch in range(n_epochs):
    # inner loop (training) - gradients ON
    loss = mini_batch(device, train_loader, train_step_fn)
    losses.append(loss)

    # VALIDATION - no gradients!
    with torch.no_grad():
        val_loss = mini_batch(device, val_loader, val_step_fn)
        val_losses.append(val_loss)

    # IMPROVEMENT: lightweight progress feedback (every 50 epochs)
    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch+1:3d} | train loss: {loss:.4f} | val loss: {val_loss:.4f}")

In [ ]:
# Did we recover the true parameters (b=1, w=2)?
print(model.state_dict())

## 5.3 Sanity check against a closed-form solution

Here is the improvement that finally uses the `LinearRegression` import. Linear regression with MSE has an exact analytic solution, so sklearn tells us the *best possible* parameters for this training split. If gradient descent worked, PyTorch's `w` and `b` must be almost identical to sklearn's. This "compare against a trusted baseline" habit is one of the most valuable debugging tools in ML engineering.


In [ ]:
# Recover the exact indices of the training split and fit sklearn on the same points
train_idx = train_data.indices
sk_model = LinearRegression().fit(x[train_idx], y[train_idx])

pt_state = model.state_dict()
print(f"sklearn  -> b = {sk_model.intercept_[0]:.4f} | w = {sk_model.coef_[0][0]:.4f}")
print(f"pytorch  -> b = {pt_state['0.bias'].item():.4f} | w = {pt_state['0.weight'].item():.4f}")
print(f"true     -> b = {true_b:.4f} | w = {true_w:.4f}")

Notice that neither solver recovers *exactly* b=1, w=2. Both recover the best fit **for this noisy sample**, and they agree with each other to several decimals. The gap to the true values is the effect of noise and finite data, not an optimization failure.

## 5.4 Loss curves


In [ ]:
fig = plt.figure(figsize=(10, 4))
plt.plot(losses, label='Training Loss', c='b')
plt.plot(val_losses, label='Validation Loss', c='r')
plt.yscale('log')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

### 📈 Reading the curves like a practitioner

- Both curves fall fast and then plateau around the noise floor. The variance of the injected noise is $0.1^2 = 0.01$, and MSE cannot go below the irreducible noise, which is why both curves flatten near $10^{-2}$.
- The log scale on the y-axis is what makes the early exponential-looking decay and the plateau readable at the same time.
- Validation slightly above training is normal and healthy. A *growing* gap over time would be the signature of overfitting.
- The wiggles come from mini-batch stochasticity and from the validation set being small (20 points).

✅ **Check yourself**

<details><summary>1. Why do the loss curves plateau near 0.01 instead of going to zero?</summary>

Because the data contains Gaussian noise with variance $0.1^2 = 0.01$. Even the perfect line leaves that residual variance behind, so 0.01 is the irreducible floor of the MSE.
</details>

<details><summary>2. Why is the validation loop wrapped in <code>torch.no_grad()</code> if <code>perform_val_step_fn</code> never calls <code>backward()</code>?</summary>

Even without `backward()`, the forward pass *records* the computation graph by default, consuming memory and time. `no_grad` skips that bookkeeping entirely. It is a performance and safety measure, not a correctness fix.
</details>

<details><summary>3. With <code>batch_size=16</code> and 80 training samples, how many parameter updates happen in 200 epochs?</summary>

5 batches per epoch × 200 epochs = 1000 updates.
</details>


# 6. Saving and Loading Models

## 6.1 What goes into a checkpoint

A trained model is more than its weights. To be able to **resume training** later, the checkpoint must capture the full training state.

- `model.state_dict()`. A dictionary mapping parameter names to tensors. This is the model's knowledge.
- `optimizer.state_dict()`. The optimizer's internal state. For plain SGD it is minimal, but for Adam it holds first and second moment estimates per parameter. Resuming Adam without them restarts the adaptive statistics from scratch and visibly perturbs training.
- Bookkeeping. Epoch counter and loss histories, so plots and schedules continue seamlessly.

Saving `state_dict`s (rather than the whole pickled model object) is the recommended practice. It is robust to refactorings of your class definitions.


In [ ]:
checkpoint = {'epoch': n_epochs,
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict(),
              'loss': losses,
              'val_loss': val_losses}

torch.save(checkpoint, 'model_checkpoint.pth')
print("checkpoint saved.")

In [ ]:
# What does the optimizer's state look like? (for SGD, mostly hyperparameters)
optimizer.state_dict()

## 6.2 Loading and resuming

Two details in the loading cell matter a lot in practice.

**`weights_only=False`.** Since PyTorch 2.6 the default flipped to `True` for security. A `.pth` file is a pickle, and unpickling arbitrary objects can execute code, so `weights_only=True` restricts loading to plain tensors. Our checkpoint holds Python lists and ints too, hence `weights_only=False`. The rule of thumb. Only do this with files **you** created or fully trust.

**`map_location=device`** (improvement). A checkpoint saved on a GPU machine fails to load on a CPU-only machine without this argument. Adding it makes the notebook portable across Colab GPU sessions and local CPUs.

Finally, **`model.train()`** after loading. If we intend to *resume training*, the model must be in training mode.


In [ ]:
checkpoint = torch.load('model_checkpoint.pth', weights_only=False, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

saved_epoch = checkpoint['epoch']
saved_losses = checkpoint['loss']
saved_val_losses = checkpoint['val_loss']

model.train()   # always use TRAIN mode when resuming training
print(f"restored at epoch {saved_epoch}")
print(model.state_dict())

✅ **Check yourself**

<details><summary>1. Why save the optimizer state if SGD "has no memory"?</summary>

Plain SGD indeed only carries hyperparameters, but the habit matters because momentum-SGD, Adam, RMSprop all keep per-parameter running statistics. Dropping them on resume changes the optimization trajectory. Saving the optimizer state makes resuming exactly equivalent to never having stopped.
</details>

<details><summary>2. What is the security concern behind <code>weights_only</code>?</summary>

`.pth` files are pickles, and pickle can execute arbitrary code during deserialization. Loading an untrusted checkpoint with `weights_only=False` is equivalent to running an untrusted script. `weights_only=True` limits deserialization to tensor data.
</details>


# 7. Deploying / Making Predictions

Inference is a different regime from training, with its own checklist.

1. Load only what inference needs, the **model** weights. No optimizer, no loss history.
2. **`model.eval()`**, always, for fully trained models. On this linear model it changes nothing, but with Dropout it prevents predictions from being randomly perturbed, and with BatchNorm it switches to the running statistics.
3. Move the inputs to the same device as the model. A device mismatch raises a `RuntimeError` immediately.
4. (Improvement) wrap the call in **`torch.no_grad()`**. Predictions do not need a computation graph, and the output arrives ready for `.cpu().numpy()` without `.detach()`.


In [ ]:
checkpoint = torch.load('model_checkpoint.pth', weights_only=False, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

new_inputs = torch.tensor([[.20], [.34], [.57]])

model.eval()   # always use EVAL for fully trained models!
with torch.no_grad():
    preds = model(new_inputs.to(device))

print(preds)
# sanity check against the true generating line (values should be close to 1 + 2x)
print("expected (noise-free):", (true_b + true_w * new_inputs).ravel().tolist())

### 🔮 Predict (pun intended)

For input `x = 0.57`, the noise-free true model gives `1 + 2(0.57) = 2.14`. The network predicted something around `2.13`. Where does the small difference come from? Answer before opening the box.

<details><summary>Answer</summary>

The network learned the best fit for a *noisy sample of 80 points*, not the true generating line. Its `b` and `w` differ from (1, 2) by a small amount (compare with the sklearn cell), so its predictions inherit that small offset. With more data, the estimates would concentrate around the true values.
</details>


# 8. 📝 Final self-test (spaced retrieval)

Close the notebook mentally and try these. Ideally revisit this section a day or two after class.

<details><summary>1. Draw the v2/v4 pipeline from memory. Which object lives on the GPU permanently, and which objects visit it?</summary>

Data Generation (NumPy) → tensors on CPU → `TensorDataset` → `random_split` → two `DataLoader`s → training loop calls `mini_batch`, which moves each batch to the device and calls a step function built by `make_train_step_fn` / `make_val_step_fn`. The **model** lives on the GPU permanently; **mini-batches** visit it one at a time.
</details>

<details><summary>2. Recite the four steps inside a training step, in order, with the PyTorch call for each.</summary>

Forward (`yhat = model(x)`), loss (`loss = loss_fn(yhat, y)`), backward (`loss.backward()`), update (`optimizer.step()` followed by `optimizer.zero_grad()`).
</details>

<details><summary>3. Which two of the four steps disappear in validation, and which two safety switches replace them?</summary>

Backward and update disappear. `model.eval()` and `torch.no_grad()` take their place, controlling layer behavior and gradient tracking respectively.
</details>

<details><summary>4. Your colleague's training loss diverges to infinity after a refactor. Name the two most likely single-line culprits from this lesson.</summary>

A missing `optimizer.zero_grad()` (gradient accumulation) or a learning rate set too high. Both produce the same symptom, an exploding loss.
</details>

<details><summary>5. Why are <code>make_train_step_fn</code> and <code>mini_batch</code> both called higher-order functions, but for different reasons?</summary>

`make_train_step_fn` *returns* a function (a closure). `mini_batch` *receives* a function (`step_fn`) as an argument. These are the two defining behaviors from the warm-up section.
</details>

# 9. 🏋️ Exercises

Do these in new cells below. Each one targets a concept from the lesson.

1. **Batch-size sweep.** Retrain with `batch_size` 4, 16, and 80 (full batch). Plot the three training curves together. Explain the differences in smoothness and in wall-clock time.
2. **Break it on purpose.** Comment out `optimizer.zero_grad()` and retrain for 50 epochs. Describe what happens to the loss and why.
3. **Learning-rate exploration.** Try `lr` in `{1.0, 0.1, 0.01, 0.001}` for 200 epochs. Which converges fastest? Which fails? Relate this to the update rule.
4. **Early stopping.** Modify the training loop to stop when the validation loss has not improved for 20 consecutive epochs, and to remember the best epoch.
5. **Closure inspection.** Print `train_step_fn.__closure__` and recover the captured `model` from it. This demystifies where the closure keeps its state.
6. **New data at inference.** Predict for `x = 2.0`, far outside the training range `[0, 1)`. Is the prediction still reasonable? What does this say about extrapolation with linear models vs neural networks in general?

# 📚 References

- Daniel Voigt Godoy, *Deep Learning with PyTorch Step-by-Step*, Chapter 2 (Rethinking the Training Loop).
- PyTorch docs. `torch.utils.data`, `torch.optim`, *Saving and Loading Models* tutorial, `torch.load` security notes.
